# 1.2 边界与状态机：失败关闭的三种姿势

**学完你能**：为 ReAct 循环写出三种失败边界的测试；用依赖注入做到「测试零成本」。

Sources: agent-expert-outline.md 思维 #6（能跑的检查）#9（预算与止损）· lecture-notes.md Slide 4/6

## 概念导读：三种边界

| 边界 | 触发 | 循环行为 | 设计思想 |
|---|---|---|---|
| 格式错误 | 模型没输出协议行 | Observation 回 `[格式错误]` | 给模型自我修正的机会，不崩溃 |
| 未知工具 | Action 里的工具名不存在 | Observation 回 `[未知工具]` | 不猜测、不静默吞掉 |
| 步数耗尽 | max_steps 内无 Finish | 返回 `[达到最大步数，未完成]` | **失败关闭**：绝不假装完成 |

这三种边界在 tests/test_react.py 各有一条测试——测试即规格。

In [ ]:
# 依赖注入：假 LLM 按剧本演出，零成本复现三种边界（对照 tests/test_react.py）
from agent.react import run_react


def scripted(*lines: str):
    it = iter(lines)
    return lambda messages: next(it)

r1 = run_react("x", {}, llm_fn=scripted("我直接回答好啦", "Thought: 好了\nFinish: done"))
r2 = run_react("x", {}, llm_fn=scripted("Thought: 试\nAction: nope x", "Thought: 停\nFinish: done"))
r3 = run_react("x", {}, llm_fn=scripted(*["Thought: 绕\nAction: calc 1"] * 20), max_steps=3)

print("边界1 观察项:", r1["steps"][0]["observation"])
print("边界2 观察项:", r2["steps"][0]["observation"])
print("边界3 答案:", r3["answer"])

In [ ]:
# 示范：注册一个新工具只需「函数 + 好描述」——ACI 第一步（ch02 会系统化）
from agent.react import run_react


def echo(arg: str) -> str:
    """原样返回输入。用于测试循环与调试协议。"""
    return f"echo: {arg}"

result = run_react("用 echo 复读一句话", {"echo": echo}, max_steps=4)
print(result["answer"])

## 练习（答案只进 solutions 子仓）

1. **改参数**：把 max_steps 与温度组合实验（0.0 / 0.7 / 1.0），各跑 3 次，记录守协议率，写 3 行结论。
2. **破坏性实验**：故意把 parse_block 的 Action 分支删掉，跑 tests/test_react.py——先红后修，体会「测试即规格」。
3. **扩展题（良好档）**：实现 timed_echo / fake_weather 任一工具 + 在 tests/regression/ 沉淀 1 条用例。

## 开源对照（作业必读）

- [mini-swe-agent](https://github.com/SWE-agent/mini-swe-agent)：约百行的真实 agent loop。对照任务与最小提交标准见 exercises.md。

## 提交前检查

```bash
uv run pytest -q && uvx ruff check .
jupyter nbconvert --clear-output --inplace 1.2-parse-and-loop.ipynb
git status
```